# Putting it together

One real problem, using everything from the first four tutorials at once. It is the program in
the [JAZ homepage](https://jaz-lang.ai)'s “Make it yours” editor with every feature ticked: the
penguins DataFrame and a web search as **inputs**, a dataclass as the **return type**, and a
**budget**, **limits** and a **log** around the whole run. Then a **validator** on top.

The question is the kind you actually get handed, with no single right answer: *how do you tell
the species apart?*

## Setup

Install JAZ, provide an API key, and pick a model. Everything below runs in Colab — no local install.

In [ ]:
%pip install -q jaz-lang pandas requests

In [ ]:
import os, getpass

if not os.environ.get('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key: ')

In [ ]:
import jaz
from jaz.llm import LiteLLM

jaz.configure(llm=LiteLLM(model='openai/gpt-5.4-nano'))

## A real tool: web search

`web_search` searches Wikipedia, with no API key. Like `same_distribution_p_value` in tutorial 1
it is a plain function, so its docstring is all the agent is told about it. The helpers above it
are plumbing: Wikipedia refuses requests' default User-Agent, so the session sends a descriptive
one, as its API policy asks.

In [ ]:
import requests
from urllib.parse import urlencode

# Wikipedia refuses requests' default User-Agent (HTTP 403).
SESSION = requests.Session()
SESSION.headers["User-Agent"] = (
    "jaz-example (https://jaz-lang.ai)")


def search_url(query: str) -> str:
    """The search, with each hit's title and lead."""
    params = urlencode({
        "action": "query", "format": "json",
        "formatversion": 2, "generator": "search",
        "gsrsearch": query, "prop": "extracts",
        "exintro": 1, "exsentences": 3,
        "exlimit": "max", "explaintext": 1,
    })
    return f"https://en.wikipedia.org/w/api.php?{params}"


def parse_hits(response: requests.Response) -> list[str]:
    """One "Title: lead" string per hit, in rank order."""
    response.raise_for_status()
    data = response.json()
    if "error" in data:
        raise RuntimeError(data["error"]["info"])
    pages = data.get("query", {}).get("pages", [])
    pages.sort(key=lambda p: p["index"])
    return [f"{p['title']}: {p.get('extract', '')}"
            for p in pages]


def web_search(query: str) -> list[str]:
    """Search Wikipedia; returns article summaries."""
    return parse_hits(SESSION.get(search_url(query)))

In [ ]:
web_search("Gentoo penguin")[0][:200]

## Everything at once

Read the call below against what you now know:

- **Inputs** (keyword arguments): `task` the prompt, `df` the DataFrame, `web_search` the tool.
  Three different kinds of object, one mechanism — tutorial 1.
- **A hook** (positional): `ReturnType(SpeciesReport)` fixes the shape of the result, and hands
  the agent the `SpeciesReport` class to build it — tutorial 3.
- **Hooks around the block** (`with`): a budget in dollars and calls, a cap on turns and on
  subagent depth, and a log file — tutorials 2 and 3. They cover every agent the call starts.

This is the homepage's program as it is, except `configure(...)`, which the setup cell already ran.

In [ ]:
from dataclasses import dataclass

import pandas as pd

from jaz import invoke
from jaz.hooks import BudgetPool, FileLogger, IterationLimit, RecursionLimit, ReturnType

PENGUINS = (
    "https://raw.githubusercontent.com"
    "/mwaskom/seaborn-data/master/penguins.csv"
)


@dataclass
class SpeciesReport:
    best_features: list[str]
    summary: str


df = pd.read_csv(PENGUINS)

with (
    BudgetPool(cost_budget=0.50, calls_budget=100),
    IterationLimit(max_iterations=10),
    RecursionLimit(max_depth=2),
    FileLogger("agent.log"),
):
    report = invoke(
        ReturnType(SpeciesReport),
        task="How to tell apart different penguin species?",
        df=df,
        web_search=web_search,
    )
print(report)

## What came back

A real `SpeciesReport`: `report.best_features` is a list you can index and loop over, not text
to parse. The summary is the model's prose, and there is no single right answer to check it
against.

`best_features` is different. It is meant to name the measurements that separate the species,
and each of those is a column of `df` — so it can be checked. That is what `ValidateReturn` is
for (tutorial 3): if a name is not a column, the agent is told which, and tries again.

In [ ]:
from jaz.hooks import ValidateReturn


def names_real_columns(report):
    """Every feature must be a column of df, so the next step can use it."""
    unknown = set(report.best_features) - set(df.columns)
    assert not unknown, f"not columns of df: {sorted(unknown)}; use the column names"


with (
    BudgetPool(cost_budget=0.50, calls_budget=100),
    IterationLimit(max_iterations=10),
    RecursionLimit(max_depth=2),
    FileLogger("agent.log"),
):
    report = invoke(
        ReturnType(SpeciesReport),
        ValidateReturn(names_real_columns),
        task="How to tell apart different penguin species?",
        df=df,
        web_search=web_search,
    )
report.best_features

Because they are real column names, checking the claim is one line of pandas. A feature that
tells the species apart should have clearly different values per species:

In [ ]:
df.groupby("species")[report.best_features].mean().round(1)

## And how it got there

The agent's answer *was* code. Every step of both runs is in `agent.log`; a `PrintLogger` shows
the same loop inline — what the model was given, the Python it wrote, and what came back. A
smaller task here, so the trace stays readable.

In [ ]:
from jaz.hooks import PrintLogger

with PrintLogger(max_field_length=160):
    gentoo = invoke(
        ReturnType(float),
        task="Compute the mean body_mass_g of Gentoo penguins.",
        df=df,
    )

gentoo

## The whole framework, in one trace

1. **The inputs were rendered into the prompt** — `task` and `df` by name, with their types. The model got the DataFrame's printed table, and the DataFrame itself in its REPL.
2. **The reply was Python, not prose.** There is nothing to parse out of it; the response *is* the program.
3. **It ran, and the agent saw the result**, ready to write another line if it had been wrong.

See inputs → write code → read output → iterate. Everything else — subagents, scope, hooks, configuration — is control over that loop.

---

## Where to go next

- [Concepts](https://jaz-lang.ai/docs/getting-started/concepts) — the same ideas, written down.
- [Hooks reference](https://jaz-lang.ai/docs/reference/hooks) — the full event and effect vocabulary.
- [Install locally](https://jaz-lang.ai/docs/getting-started/installation).